# C7三图专项补标：待另行授权的板测入口

本任务只完成上板前准备。当前不授权加载位流。只处理 im0004、im0291、im0319 的原512输入；旧C7掩膜来自已核验PL归档。新PL从RGB计算新增，不播放参考mask。

不要 Run All。不要在旧故障内核中运行，不释放旧缓冲。本页打开、解压及离线检查不访问硬件；先另行确认旧现场安全保存及本版本板测授权。

In [ ]:
from pathlib import Path
import zipfile
archive=Path.cwd()/'C7_ThreeImage_Completion_Board_r1.zip'
assert archive.is_file()
assert not (Path.cwd()/'payload').exists(), '目录已存在；保留原目录，不覆盖解压'
with zipfile.ZipFile(archive) as z:
    root_dir=Path.cwd().resolve()
    for name in z.namelist():
        target=(root_dir/name).resolve()
        assert root_dir in target.parents, '不安全路径'
    for name in z.namelist():
        if Path(name).parts and Path(name).parts[0]=='payload':
            z.extract(name,root_dir)
print('仅解压完成，未访问板卡')


In [ ]:
import sys
root=Path.cwd()/'payload'
sys.path.insert(0,str(root))
import board_run
manifest=board_run.verify(root)
print('OFFLINE_PACKAGE_VERIFIED',manifest['algorithm'],manifest['case_order'])
print('此格只核验文件；尚未执行板测')


## 获得本版本单独板测授权后才可运行

当前保持AUTHORIZED=False。另行授权且人工确认板卡空闲、旧现场安全保存之后，才能手动改为True。此格加载新50MHz精修overlay，先3个小工作区配置测试，再依次执行三张真实图。失败立即停止并保留状态，不自动重试、释放或复位。每张图可能数十秒，总流程可能数分钟；最终实际时间以记录为准。

In [ ]:
AUTHORIZED=False
assert AUTHORIZED, '当前仅上板前交付；须另行授权并人工确认后执行'
result=board_run.start(root,Path.cwd()/'three_image_board_results_r1',
    operator_confirmation='I_AUTHORIZE_THREE_IMAGE_BOARD_TEST_ONCE')
print(result)


## 仅在全部运行成功后显示

下格只读取刚保存的真实PL结果。图片顺序：原图、原C7、原绿/新增青。预期输出仅用于逐字核对，不写入PL。

In [ ]:
from PIL import Image
from IPython.display import display
for image_path in board_run.render_saved(root,Path.cwd()/'three_image_board_results_r1'):
    print(image_path)
    display(Image.open(image_path))


## 导出记录

完成或报错后、内核仍可响应时，只运行此格导出。若In[*]或页面断连，不追加运行、不重启或断电；先保留当前页面与记录。

In [ ]:
from IPython.display import FileLink,display
p=Path.cwd()/'three_image_board_results_r1.zip'
print(board_run.export_results(p))
display(FileLink(p.name))
